# 🩺 Skin Disease Classification — HAM10000
**Model**: EfficientNetB3 (Transfer Learning)  
**Dataset**: HAM10000 — 10,015 dermoscopic images across 7 classes

---

In [ ]:
# Install dependencies (run once)
# !pip install tensorflow keras scikit-learn pandas matplotlib seaborn opencv-python Pillow

In [ ]:
import os, sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_style('darkgrid')

## 1. Load & Explore the Dataset

In [ ]:
# Update these paths after downloading the dataset
CSV_PATH  = '../data/HAM10000_metadata.csv'
IMG_DIR   = '../data/'

LABEL_MAP = {
    'nv': 'Melanocytic Nevi', 'mel': 'Melanoma',
    'bkl': 'Benign Keratosis', 'bcc': 'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis', 'vasc': 'Vascular Lesions', 'df': 'Dermatofibroma'
}

df = pd.read_csv(CSV_PATH)
df['label_name'] = df['dx'].map(LABEL_MAP)
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
counts = df['label_name'].value_counts()

colors = plt.cm.Set3(np.linspace(0, 1, len(counts)))
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_xticklabels(counts.index, rotation=45, ha='right')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Class Proportions')
plt.tight_layout()
plt.show()
print(counts)

In [ ]:
# Sample images per class
CLASS_NAMES = list(LABEL_MAP.values())
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    subset = df[df['label_name'] == cls]
    row = subset.sample(1).iloc[0]
    img_path = None
    for sub in ['HAM10000_images_part_1', 'HAM10000_images_part_2', '']:
        p = os.path.join(IMG_DIR, sub, f"{row['image_id']}.jpg")
        if os.path.exists(p):
            img_path = p; break
    if img_path:
        axes[i].imshow(Image.open(img_path))
        axes[i].set_title(cls, fontsize=9)
    axes[i].axis('off')

axes[-1].axis('off')
plt.suptitle('One Sample per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demographics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='label_name', y='age', ax=axes[0], palette='Set3')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_title('Age Distribution by Class')

sex_data = pd.crosstab(df['label_name'], df['sex'])
sex_data.plot.bar(ax=axes[1], stacked=True, color=['#64b5f6', '#f48fb1', '#a5d6a7'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Sex Distribution by Class')

plt.tight_layout()
plt.show()

## 2. Build & Train the Model

In [ ]:
from src.train import build_generators, build_model, unfreeze_top_layers, get_callbacks
from src.train import load_metadata, get_class_weights

df_full = load_metadata(CSV_PATH, IMG_DIR)
train_gen, val_gen, test_gen, train_df = build_generators(df_full)
class_weights = get_class_weights(train_df['label_idx'].values)

print(f'Train batches: {len(train_gen)}')
print(f'Val   batches: {len(val_gen)}')
print(f'Test  batches: {len(test_gen)}')

In [ ]:
model, base_model = build_model()
model.summary()

In [ ]:
# Phase 1: Train head only
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=get_callbacks(1, '../models'),
)

In [ ]:
# Phase 2: Fine-tune top layers
model = unfreeze_top_layers(model, base_model)
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=35,
    class_weight=class_weights,
    callbacks=get_callbacks(2, '../models'),
    initial_epoch=len(history1.history['loss']),
)

## 3. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_gen.reset()
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Single Image Inference

In [ ]:
from src.predict import predict

# Test on a single image
test_image_path = 'path/to/your/test_image.jpg'  # <-- Update this

result = predict(test_image_path, model_path='../models/skin_disease_final.h5')

print(f"Predicted:   {result['predicted_class']}")
print(f"Confidence:  {result['confidence']*100:.1f}%")
print(f"Risk:        {result['info']['risk']}")
print(f"Action:      {result['info']['action']}")
print("\nTop-3 Predictions:")
for r in result['top_k']:
    print(f"  {r['class']:25s} {r['confidence']*100:5.1f}%")

In [ ]:
# Visualize prediction with bar chart
probs = result['all_probabilities']
classes = list(probs.keys())
values  = [probs[c]*100 for c in classes]

plt.figure(figsize=(10, 5))
bars = plt.barh(classes, values, color=['#22c55e' if v < 50 else '#ef4444' for v in values])
plt.xlabel('Confidence (%)')
plt.title(f"Prediction: {result['predicted_class']} ({result['confidence']*100:.1f}%)")
plt.tight_layout()
plt.show()